In [119]:
import json
import re
from typing import Any, TypedDict
import bs4
import requests


class Expr(TypedDict):
    name: str
    args: list["Expr"]


def parse(s: str) -> list[Expr]:
    stack: list[list[Expr]] = [[]]
    can_accept_args = False
    while s:
        if match := re.match(r"\s+", s):
            pass
        elif match := re.match(r"#.*", s):
            pass
        elif match := re.match(r"\(", s):
            if not can_accept_args:
                raise Exception("invalid arg list")
            stack.append([])
            can_accept_args = False
        elif match := re.match(r"\)", s):
            if len(stack) <= 1:
                raise Exception("unbalanced parentheses")
            child = stack.pop()
            stack[-1][-1]["args"] = child
            can_accept_args = False
        elif match := re.match(r"[\w\d\-_\/\.:\?\!=]+", s):
            stack[-1].append({"name": match.group(), "args": []})
            can_accept_args = True
        elif match := re.match(r"\"(\\\"|[^\"])*\"|\'(\\\'|[^\'])*\'", s):
            stack[-1].append({"name": eval(match.group()), "args": []})
            can_accept_args = True
        else:
            raise Exception(f"failed to tokenize: {repr(s)}")
        s = s[match.end() :]
    if len(stack) != 1:
        raise Exception("unbalanced parentheses")
    return stack[0]


type Value = Any


def evaluate_proc(proc: Expr, inp: Value) -> Value:
    proc_name = proc["name"]
    proc_args = proc["args"]
    if proc_name == "json":
        assert len(proc_args) == 0
        return json.loads(inp)
    elif proc_name == "read":
        assert len(proc_args) == 0
        assert type(inp) == str
        with open(inp, "r") as f:
            return f.read()
    elif proc_name == "do":
        return evaluate_procs(proc_args, inp)
    elif proc_name == "id":
        assert len(proc_args) == 0
        return inp
    elif proc_name == "array":
        result = []
        for elt in proc_args:
            result.append(evaluate_proc(elt, inp))
        return result
    elif proc_name == "fetch":
        assert len(proc_args) == 0
        assert type(inp) == str
        response = requests.get(
            inp,
            headers={
                "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"
            },
        )
        if not response.ok:
            raise Exception(f"request error: {inp} -> {response.status_code}")
        return response.text
    elif proc_name == "str":
        if len(proc_args) == 0:
            assert type(inp) in [int, float, str, bool]
            if type(inp) == bool:
                return str(inp).lower()
            else:
                return str(inp)
        else:
            assert len(proc_args) == 1
            assert len(proc_args[0]["args"]) == 0
            return proc_args[0]["name"]
    elif proc_name == "trim":
        assert len(proc_args) == 0
        assert type(inp) == str
        return inp.strip()
    elif proc_name == "rmPrefix":
        assert len(proc_args) == 1
        assert len(proc_args[0]["args"]) == 0
        assert type(inp) == str
        assert inp.startswith(proc_args[0]["name"])
        return inp.removeprefix(proc_args[0]["name"])
    elif proc_name == "rmSuffix":
        assert len(proc_args) == 1
        assert len(proc_args[0]["args"]) == 0
        assert type(inp) == str
        assert inp.endswith(proc_args[0]["name"])
        return inp.removesuffix(proc_args[0]["name"])
    elif proc_name == "join":
        if len(proc_args) == 0:
            sep = ""
        else:
            assert len(proc_args) == 1
            assert len(proc_args[0]["args"]) == 0
            sep = proc_args[0]["name"]
        assert type(inp) == list
        for item in inp:
            assert type(item) == str
        return sep.join(inp)
    elif proc_name == "eq":
        assert len(proc_args) == 1
        assert len(proc_args[0]["args"]) == 0
        assert type(inp) == str
        return inp == proc_args[0]["name"]
    elif proc_name == "not":
        assert len(proc_args) == 0
        assert type(inp) == bool
        return not inp
    elif proc_name == "index":
        assert len(proc_args) == 1
        assert len(proc_args[0]["args"]) == 0
        assert type(inp) == list
        index = int(proc_args[0]["name"])
        assert index >= -len(inp) and index < len(inp)
        return inp[index]
    elif proc_name == "index?":
        assert len(proc_args) == 1
        assert len(proc_args[0]["args"]) == 0
        assert type(inp) == list
        index = int(proc_args[0]["name"])
        if index >= -len(inp) and index < len(inp):
            return inp[index]
        else:
            return None
    elif proc_name == "slice":
        assert len(proc_args) == 1 or len(proc_args) == 2
        assert type(inp) == list
        assert len(proc_args[0]["args"]) == 0
        start = int(proc_args[0]["name"])
        if len(proc_args) == 1:
            end = len(inp)
        else:
            assert len(proc_args[1]["args"]) == 0
            end = int(proc_args[1]["name"])
        start = max(start, 0)
        end = min(end, len(inp))
        return inp[start:end]
    elif proc_name == "flatten":
        assert len(proc_args) == 0
        assert type(inp) == list
        result = []
        for item in inp:
            assert type(item) == list
            result += item
        return result
    elif proc_name == "range":
        assert len(proc_args) == 1 or len(proc_args) == 2
        assert len(proc_args[0]["args"]) == 0
        if len(proc_args) == 1:
            start = 0
            end = int(proc_args[0]["name"])
        else:
            assert len(proc_args[1]["args"]) == 0
            start = int(proc_args[0]["name"])
            end = int(proc_args[1]["name"])
        return list(range(start, end))
    elif proc_name == "field":
        assert len(proc_args) == 1
        assert len(proc_args[0]["args"]) == 0
        assert type(inp) == dict
        return inp[proc_args[0]["name"]]
    elif proc_name == "field?":
        assert len(proc_args) == 1
        assert len(proc_args[0]["args"]) == 0
        assert type(inp) == dict
        return inp.get(proc_args[0]["name"])
    elif proc_name == "count":
        assert len(proc_args) == 0
        assert type(inp) == list
        return len(inp)
    elif proc_name == "map":
        assert type(inp) == list
        return [evaluate_procs(proc_args, item) for item in inp]
    elif proc_name == "filter":
        assert type(inp) == list
        result = []
        for item in inp:
            cond = evaluate_procs(proc_args, item)
            assert type(cond) == bool
            if cond:
                result.append(item)
        return result
    elif proc_name == "fromEntries":
        assert len(proc_args) == 0
        assert type(inp) == list
        result = {}
        for entry in inp:
            assert len(entry) == 2
            assert type(entry[0]) == str
            result[entry[0]] = entry[1]
        return result
    elif proc_name == "groupBy":
        assert type(inp) == list
        result = {}
        for item in inp:
            key = evaluate_procs(proc_args, item)
            assert type(key) == str
            result.setdefault(key, []).append(item)
        return result
    elif proc_name == "keyBy":
        assert type(inp) == list
        result = {}
        for item in inp:
            key = evaluate_procs(proc_args, item)
            assert type(key) == str
            assert key not in result
            result[key] = item
        return result
    elif proc_name == "mapValues":
        assert type(inp) == dict
        return {key: evaluate_procs(proc_args, value) for key, value in inp.items()}
    elif proc_name == "int":
        assert len(proc_args) == 0
        assert type(inp) in [str, int]
        return int(inp)
    elif proc_name == "float":
        assert len(proc_args) == 0
        assert type(inp) in [str, int, float]
        return float(inp)
    elif proc_name == "sum":
        assert len(proc_args) == 0
        assert type(inp) == list
        total = 0
        for item in inp:
            assert type(item) in [int, float]
            total += item
        return total
    elif proc_name == "average":
        assert len(proc_args) == 0
        assert type(inp) == list
        total = 0
        for item in inp:
            assert type(item) in [int, float]
            total += item
        return total / len(inp)
    elif proc_name == "html":
        assert len(proc_args) == 0
        assert type(inp) == str
        return bs4.BeautifulSoup(inp, "html.parser")
    elif proc_name == "html.select":
        assert len(proc_args) == 1
        assert len(proc_args[0]["args"]) == 0
        assert isinstance(inp, bs4.Tag)
        return list(inp.select(proc_args[0]["name"]))
    elif proc_name == "html.text":
        assert len(proc_args) == 0
        assert isinstance(inp, bs4.Tag)
        return inp.text
    else:
        raise Exception(f"unknown proc: {proc}")


def evaluate_procs(procs: list[Expr], inp: Value) -> Value:
    for proc in procs:
        inp = evaluate_proc(proc, inp)
    return inp


def evaluate(s: str, inp: Value | None = None) -> Value:
    procs = parse(s)
    return evaluate_procs(procs, inp)


n_pages = evaluate(r"""
str(https://tryscrapeme.com/web-scraping-practice/beginner/pagination)
fetch html
html.select(".pagination li")
index(-1) html.text int
""")

s = rf"""
range(1 {n_pages + 1})
map(str array(
    str(https://tryscrapeme.com/web-scraping-practice/beginner/pagination?pageno=)
    id
) join)
map(
    fetch html
    html.select("tbody tr")
    map(html.select(td) index(3) html.text float)
)
flatten
sum
"""

result = evaluate(s)
print(result)
# print(json.dumps(result, indent=2))

724.84


In [150]:
re.match(r"[-+]?(e[-+]?|[\.\w\d_])+", "5e1")

<re.Match object; span=(0, 3), match='5e1'>